# PLACES — A Number That Was Never Measured

Every source this corpus has met so far reports something that was *observed*: a household
answered a survey, an employer filed a return, an administrator set a rule. A CDC PLACES
county estimate is different in kind. It is the output of a multilevel model fitted to BRFSS
and poststratified against Census population and ACS covariates — **a prediction about the
county, not a measurement taken in it.** A county may contribute no BRFSS respondents at all
and still receive an estimate to one decimal place, with an interval.

That distinction is not pedantry, and this notebook is organised around what follows from it.
Two of the sections below could not be written about any other source in this corpus, because
the questions they ask only arise once the number is modelled.

## Learning Objectives

1. **Distinguish a modelled interval from a sampling margin empirically**, not by reading the
   documentation — by measuring how each scales with population.
2. **Separate reliability from rank identifiability**, and see a measure clear the conventional
   publishing threshold while still supporting no ranking whatsoever.
3. **Replace an impossible estimand with possible ones** — detection, tiers, exceedance — and
   quantify what each actually yields.
4. **Bound the new information in a ranking** by asking how much of it survives without the
   modelled source at all.

## Prerequisites

- Handbook §1 (the data regimes) and §7.1–§7.2 (rankings, and what a ranking may claim).
- D01 or D02 for the ACS margin-of-error conventions this notebook compares against.
- Comfort with the idea of a standard error; no Bayesian background is assumed.

## Data Sources

| source | what it is | vintage | credential |
|---|---|---|---|
| CDC PLACES, county (`i46a-9kgh`) | **model-based** prevalence from BRFSS, poststratified | 2025 release | none |
| Census ACS 5-year, `B17001` | design-based poverty counts with published MOE | 2023 | `CENSUS_API_KEY` |
| Census ACS 5-year, age/income/poverty | covariates for the §6 recoverability bound | 2023 | `CENSUS_API_KEY` |

Every retrieval writes a provenance record to `provenance/` carrying the endpoint, the
retrieval time, the row and column counts, and a SHA-256 of the exact payload.

In [1]:
"""Configuration. Every analytic choice that follows is named here, not inferred."""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as st
import matplotlib.pyplot as plt

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src"))
        break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data.sources import fetch_places_county, fetch_acs

STATE = "Virginia"
STATE_FIPS = "51"
PLACES_RELEASE = "2025"       # each release is a separate asset; naming one pins the vintage
ADJUSTMENT = "crude"          # crude counts burden in one place; age-adjusted compares places
ACS_YEAR = 2023
MEASURES = ["diabetes", "casthma", "obesity", "depression"]
SEED = 20260914
Z95 = 1.959964
Z998 = 3.090232
#: The repository root, found by walking UP from wherever this runs. Provenance
#: used to be written to `Path.cwd() / "provenance"` in some notebooks and to a
#: repo-root-relative path in others, so the two conventions could not both be
#: satisfied by one working directory -- and a run from the wrong one silently
#: wrote sixteen records into a stray folder at the top of the repo.
REPO_ROOT = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (c / "src" / "krl_tutorials_data").is_dir())

OUT = REPO_ROOT / "projects" / "02-health-housing/D04_health_prevalence" / "provenance"
print(f"PLACES {PLACES_RELEASE} release, {ADJUSTMENT} prevalence, {STATE}")
print(f"measures: {', '.join(MEASURES)}")

PLACES 2025 release, crude prevalence, Virginia
measures: diabetes, casthma, obesity, depression


In [2]:
"""Retrieve PLACES and pin the provenance."""
places, prov_places = fetch_places_county(
    measures=MEASURES, release=PLACES_RELEASE, adjustment=ADJUSTMENT, state=STATE
)
places["fips"] = places["countyfips"]
places["pop"] = pd.to_numeric(places["totalpopulation"], errors="coerce")
written = [prov_places.write(OUT / "d04_places_county.json")]

print(prov_places.render())
print()
print(f"{len(places)} counties and independent cities in {STATE}")

Source      : Centers for Disease Control and Prevention (PLACES) - PLACES county estimates, 2025 release, crude prevalence
Vintage     : 2025  (pinned)
Endpoint    : https://data.cdc.gov/resource/i46a-9kgh.json?%24select=countyfips%2Ccountyname%2Cstatedesc%2Ctotalpopulation%2Cdiabetes_crudeprev%2Cdiabetes_crude95ci%2Ccasthma_crudeprev%2Ccasthma_crude95ci%2Cobesity_crudeprev%2Cobesity_crude95ci%2Cdepression_crudeprev%2Cdepression_crude95ci&%24limit=5000&%24where=statedesc%3D%27Virginia%27
Retrieved   : 2026-09-18T12:12:58+00:00
Shape       : 133 rows x 28 cols
Payload hash: b7f44c7995992209...  (canonical-json)
Data mode   : EMPIRICAL
Variables   :
              diabetes         diabetes crude prevalence and 95% interval
              casthma          casthma crude prevalence and 95% interval
              obesity          obesity crude prevalence and 95% interval
              depression       depression crude prevalence and 95% interval
Limitations :
              - MODEL-BASED estim

## 1. What the number is, and what its two versions are for

PLACES publishes each measure twice — crude and age-adjusted — under column names one token
apart. They are not two precisions of the same quantity. **Crude prevalence counts the burden
that actually exists in a place**, which is what a clinic sizing its caseload needs.
**Age-adjusted prevalence removes the age structure** so that two places can be compared
without the older one appearing sicker for that reason alone.

Choosing wrongly does not raise anything. It quietly answers a different question.

In [3]:
"""How far apart are the two versions, and does the choice change the ordering?"""
adj, _ = fetch_places_county(
    measures=MEASURES, release=PLACES_RELEASE, adjustment="adj", state=STATE
)
adj["fips"] = adj["countyfips"]

rows = []
for m in MEASURES:
    paired = places[["fips", f"{m}_prev"]].merge(
        adj[["fips", f"{m}_prev"]], on="fips", suffixes=("_crude", "_adj")
    ).dropna()
    gap = paired[f"{m}_prev_crude"] - paired[f"{m}_prev_adj"]
    rho = st.spearmanr(paired[f"{m}_prev_crude"], paired[f"{m}_prev_adj"]).statistic
    rows.append({
        "measure": m,
        "median gap (pp)": round(float(gap.median()), 2),
        "max |gap| (pp)": round(float(gap.abs().max()), 2),
        "rank corr": round(float(rho), 4),
    })
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
print()
print("This does not say what it was expected to say, and the exception is the point.")
print("Asthma and obesity reorder barely (rank corr 0.95+): for them the choice moves the")
print("LEVEL and leaves the ORDER. Diabetes does not -- rank corr 0.68, with a median gap")
print("of about 3 points. Age structure is doing real work there, so for diabetes the two")
print("versions are not one quantity at two calibrations; they answer different questions")
print("and disagree about which counties are worst. The choice cannot be delegated to a")
print("default, and it cannot be assumed harmless because it was harmless for asthma.")

   measure  median gap (pp)  max |gap| (pp)  rank corr
  diabetes              2.9             6.5     0.6751
   casthma             -0.1             0.6     0.9731
   obesity              0.1             6.6     0.9469
depression             -1.2             3.8     0.8759

This does not say what it was expected to say, and the exception is the point.
Asthma and obesity reorder barely (rank corr 0.95+): for them the choice moves the
LEVEL and leaves the ORDER. Diabetes does not -- rank corr 0.68, with a median gap
of about 3 points. Age structure is doing real work there, so for diabetes the two
versions are not one quantity at two calibrations; they answer different questions
and disagree about which counties are worst. The choice cannot be delegated to a
default, and it cannot be assumed harmless because it was harmless for asthma.


## 2. Precision that does not come from sample size

This is the section that could not be written about any other source here, and it is the
cleanest available demonstration that a modelled interval is a different object from a
sampling margin.

Design theory is specific. For a quantity estimated from a sample drawn in the place itself,
the standard error of a **rate** falls with the square root of the sample, and the sample
scales with the population: slope of −0.5 in logs. The standard error of a **count** rises
with the square root: +0.5.

A modelled estimate has no such obligation, because its precision comes from the model, not
from local respondents — of whom there may be none. If that is true, PLACES intervals should
be roughly **flat** in population where ACS margins are not.

Rather than assert it, measure both on the same 133 counties.

In [4]:
"""Do the two kinds of uncertainty scale with population the same way?"""
poverty, prov_acs = fetch_acs(
    year=ACS_YEAR,
    variables={"B17001_001E": "poverty universe", "B17001_001M": "poverty universe MOE",
               "B17001_002E": "persons in poverty", "B17001_002M": "persons in poverty MOE"},
    geography="county:*", within=f"state:{STATE_FIPS}",
)
poverty["fips"] = poverty["state"] + poverty["county"]
written.append(prov_acs.write(OUT / "d04_acs_b17001_county.json"))
merged = places.merge(poverty, on="fips")

def log_slope(x, y) -> tuple[float, int]:
    """d log(y) / d log(x), fitted on the counties where both are positive and finite."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    return float(np.polyfit(np.log(x[ok]), np.log(y[ok]), 1)[0]), int(ok.sum())

acs_se = merged["B17001_002M"] / 1.645          # ACS publishes a 90% MOE
universe = merged["B17001_001E"]

print("d log(standard error) / d log(population)\n")
print("  ACS, drawn from a sample in the county itself:")
for label, series, theory in [
    ("standard error of the COUNT", acs_se, "+0.5"),
    ("standard error of the RATE ", acs_se / universe, "-0.5"),
]:
    s, n = log_slope(universe, series)
    print(f"    {label}   {s:+.3f}   design theory {theory}   (n={n})")

print("\n  PLACES, modelled:")
for m in MEASURES:
    s, n = log_slope(merged["pop"], merged[f"{m}_se"])
    print(f"    {m:26}   {s:+.3f}   (n={n})")

d log(standard error) / d log(population)

  ACS, drawn from a sample in the county itself:
    standard error of the COUNT   +0.523   design theory +0.5   (n=133)
    standard error of the RATE    -0.477   design theory -0.5   (n=133)

  PLACES, modelled:
    diabetes                     -0.087   (n=133)
    casthma                      -0.029   (n=133)
    obesity                      -0.109   (n=133)
    depression                   -0.044   (n=133)


The ACS rate reproduces design theory almost exactly. PLACES does not come close to it, and
that is not a defect — it is the signature of a different kind of estimate. **A county of five
thousand people and a county of a million receive intervals of similar width**, which is
impossible for a quantity measured by sampling that county.

The practical consequence is that a reader cannot use interval width as a proxy for how much
local evidence stands behind a PLACES number. There may be none. Width reflects where the
county sits in the model's covariate space, not how many people were asked.

## 3. The question the data cannot answer

Handbook §7.1 measured 13,981 census tracts and found not one with a determined rank. The
obvious reading was that tracts are simply too small. PLACES tests that reading at county
level — two steps coarser — and §7.2 records the result across all 2,956 US counties.

Here the same machinery runs on Virginia alone, where a reader can see the counties by name.
Two quantities matter and they are routinely confused:

- **Reliability** `λ = τ² / (τ² + se²)` asks whether a county can be told apart *from the
  average*. It is the shrinkage weight in a hierarchical model, and provider profiling
  conventionally requires about 0.7 before publishing unit-level comparisons.
- **Rank identifiability** asks whether a county can be told apart *from every other county*.

Clearing the first says nothing about the second.

One further caution before the numbers: **reliability is not a property of a measure.** It is
a property of a measure *and the set it is compared within*. Handbook §7.2 reports a pooled
λ of 0.43 for asthma across all 2,956 US counties; within Virginia alone it collapses, because
τ² — the genuine spread between units — is far smaller inside one state while the interval
widths are unchanged. A measure can be reliable nationally and useless within a state.

In [5]:
"""Reliability, then the ranking attempt it does not license."""
summary = []
for m in MEASURES:
    est = places[f"{m}_prev"].to_numpy(float)
    se = places[f"{m}_se"].to_numpy(float)
    lo, hi = places[f"{m}_lo"].to_numpy(float), places[f"{m}_hi"].to_numpy(float)
    ok = np.isfinite(est) & np.isfinite(se) & (se > 0)
    est, se, lo, hi = est[ok], se[ok], lo[ok], hi[ok]

    # tau^2 by the DerSimonian-Laird moment estimator: observed spread minus
    # mean sampling variance, floored at zero.
    tau2 = max(float(est.var(ddof=1) - (se ** 2).mean()), 0.0)
    lam = tau2 / (tau2 + se ** 2)

    determined = sum(
        1 for i in range(len(est))
        if not ((lo < hi[i]) & (hi > lo[i]) & (np.arange(len(est)) != i)).any()
    )
    # Deterministic rank bounds, not a simulation. For county i the best rank it
    # could hold is one more than the number of counties whose interval lies
    # ENTIRELY above it; the worst is n minus the number lying entirely below.
    # Everything else overlaps i and could fall either side. This assumes nothing
    # about the shape of the interval -- unlike §7.2's Monte Carlo, which needs
    # normality -- and it cannot be narrower than the truth.
    best = np.array([1 + int((lo > hi[i]).sum()) for i in range(len(est))])
    worst = np.array([len(est) - int((hi < lo[i]).sum()) for i in range(len(est))])
    width = worst - best

    summary.append({
        "measure": m, "n": len(est),
        "lambda (pooled)": round(tau2 / (tau2 + float((se ** 2).mean())), 3),
        "share lambda>=0.7": f"{100 * (lam >= 0.7).mean():.0f}%",
        "determined ranks": determined,
        "rank span (median)": f"{np.median(width):.0f} of {len(est)}",
    })
ranking = pd.DataFrame(summary)
print(ranking.to_string(index=False))

   measure   n  lambda (pooled) share lambda>=0.7  determined ranks rank span (median)
  diabetes 133            0.853              100%                 0          97 of 133
   casthma 133            0.071                0%                 0         132 of 133
   obesity 133            0.113                0%                 0         130 of 133
depression 133            0.117                0%                 0         131 of 133


Diabetes is the decisive row. Its reliability clears the conventional threshold for every
county in the state — and it still supports **no** determined rank, with a median rank
interval covering a large fraction of the list.

That is not a precision problem, so precision cannot fix it. Telling one county from ~130
others requires every pairwise comparison to separate at once, and the number of comparisons
grows with the square of the list. Reliability and rank are different questions, and the
second fails on multiplicity at any precision a real survey will reach.

**[rule]** Do not rank places on a modelled small-area estimate. §7.2 gives the general
result; the next two sections give the estimands to use instead.

## 4. Detection instead of ordering

If "which county is worst?" is unanswerable, "which counties are unusual?" is not. Compare
each county against a precision-weighted benchmark, with limits that **widen for imprecise
units by construction** — so small counties are not thrown into the tails mechanically, which
is the failure Gelman & Price (1999) describe for maps of parameter estimates.

In [6]:
"""Funnel: how many counties are genuinely unusual, where ranking identified none?"""
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
detected = []
for ax, m in zip(axes, ["diabetes", "casthma"]):
    est = places[f"{m}_prev"].to_numpy(float)
    se = places[f"{m}_se"].to_numpy(float)
    pop = places["pop"].to_numpy(float)
    ok = np.isfinite(est) & np.isfinite(se) & (se > 0) & np.isfinite(pop)
    est, se, pop = est[ok], se[ok], pop[ok]

    w = 1 / se ** 2
    theta0 = float((w * est).sum() / w.sum())
    z = (est - theta0) / se
    detected.append({
        "measure": m, "benchmark": round(theta0, 2),
        "outside 95%": f"{100 * (np.abs(z) > Z95).mean():.1f}%",
        "outside 99.8%": f"{100 * (np.abs(z) > Z998).mean():.1f}%",
    })

    grid = np.linspace(se.min(), se.max(), 200)
    ax.scatter(se, est, s=14, alpha=0.55, color="#4C72B0", label="county")
    for zc, style in [(Z95, "--"), (Z998, ":")]:
        ax.plot(grid, theta0 + zc * grid, style, color="#666", lw=1)
        ax.plot(grid, theta0 - zc * grid, style, color="#666", lw=1)
    ax.axhline(theta0, color="#C44E52", lw=1.2)
    ax.set_xlabel("standard error  (imprecise ->)")
    ax.set_ylabel(f"{m} prevalence (%)")
    ax.set_title(f"{m}: benchmark {theta0:.1f}%, 95% and 99.8% limits")
fig.suptitle(f"Funnel, {STATE} counties, PLACES {PLACES_RELEASE} {ADJUSTMENT}", y=1.02)
fig.tight_layout()
plt.show()

print(pd.DataFrame(detected).to_string(index=False))
print()
print("Ranking identified 0 counties. This identifies a definite, nameable set.")

 measure  benchmark outside 95% outside 99.8%
diabetes      13.25       48.1%         30.1%
 casthma      10.88        8.3%          2.3%

Ranking identified 0 counties. This identifies a definite, nameable set.


## 5. Two more estimands the data will support

**Tiers.** Walk the counties in descending order and open a new group only where the next
county sits significantly below the current group's leader, Bonferroni-corrected. The number
of groups that survive is the resolution the measurement actually has.

**Exceedance.** `P(θ > c)` for a threshold `c` is a well-posed question about one county at a
time, with no multiplicity penalty. It carries an asymmetry worth exploiting.

In [7]:
"""Tiers and exceedance."""
rows = []
for m in MEASURES:
    est = places[f"{m}_prev"].to_numpy(float)
    se = places[f"{m}_se"].to_numpy(float)
    ok = np.isfinite(est) & np.isfinite(se) & (se > 0)
    est, se = est[ok], se[ok]

    order = np.argsort(-est)
    e, s = est[order], se[order]
    z_crit = float(st.norm.ppf(1 - (0.05 / max(len(e) - 1, 1)) / 2))
    n_tiers, leader = 1, 0
    for i in range(1, len(e)):
        if (e[leader] - e[i]) / np.hypot(s[leader], s[i]) > z_crit:
            n_tiers += 1
            leader = i

    c = float(np.percentile(est, 90))
    p = 1 - st.norm.cdf((c - est) / se)
    rows.append({
        "measure": m,
        "tiers": n_tiers,
        "threshold c (90th pct)": round(c, 1),
        "confidently above": f"{100 * (p > 0.95).mean():.1f}%",
        "confidently below": f"{100 * (p < 0.05).mean():.1f}%",
        "indeterminate": f"{100 * ((p >= 0.05) & (p <= 0.95)).mean():.1f}%",
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Note the asymmetry: far more counties can be confidently EXCLUDED from the high-burden")
print("set than confidently placed in it. A screening list built by exclusion rests on firmer")
print("evidence than one built by selection -- the opposite of how such lists are usually made.")

   measure  tiers  threshold c (90th pct) confidently above confidently below indeterminate
  diabetes      3                    18.2              0.0%             73.7%         26.3%
   casthma      2                    11.7              0.0%             24.1%         75.9%
   obesity      2                    42.6              0.0%             34.6%         65.4%
depression      2                    25.8              0.8%             43.6%         55.6%

Note the asymmetry: far more counties can be confidently EXCLUDED from the high-burden
set than confidently placed in it. A screening list built by exclusion rests on firmer
evidence than one built by selection -- the opposite of how such lists are usually made.


## 6. How much of this needed PLACES at all?

A modelled estimate is built from covariates. If a ranking of counties on the estimate is
largely reproducible from those covariates directly, then the ranking carries less new
information than its decimal places suggest.

This is a bound, not an accusation. A high overlap does not establish which covariates the
model used, and poverty genuinely does predict diabetes — the association may be entirely
substantive. What it does establish is how much a reader who already holds ACS would learn
from the ranking.

In [8]:
"""Regress each PLACES estimate on four ACS covariates and compare the orderings."""
cov, prov_cov = fetch_acs(
    year=ACS_YEAR,
    variables={"B01002_001E": "median age", "B19013_001E": "median household income",
               "B17001_001E": "poverty universe", "B17001_002E": "persons in poverty",
               "B01001_001E": "total population", "B01001_020E": "male 65-66",
               "B01001_044E": "female 65-66"},
    geography="county:*", within=f"state:{STATE_FIPS}",
)
cov["fips"] = cov["state"] + cov["county"]
written.append(prov_cov.write(OUT / "d04_acs_covariates_county.json"))
for c in ["B01002_001E", "B19013_001E", "B17001_001E", "B17001_002E",
          "B01001_001E", "B01001_020E", "B01001_044E"]:
    cov[c] = pd.to_numeric(cov[c], errors="coerce")
    cov.loc[cov[c] < 0, c] = np.nan     # Census jam values are annotations, not data
cov["poverty_rate"] = 100 * cov.B17001_002E / cov.B17001_001E
cov["pct65"] = 100 * (cov.B01001_020E + cov.B01001_044E) / cov.B01001_001E

rows = []
for m in MEASURES:
    frame = places[["fips", f"{m}_prev"]].merge(
        cov[["fips", "B01002_001E", "B19013_001E", "poverty_rate", "pct65"]], on="fips"
    ).dropna()
    frame = frame[frame.B19013_001E > 0]
    X = np.column_stack([np.ones(len(frame)), frame.B01002_001E,
                         np.log(frame.B19013_001E), frame.poverty_rate, frame.pct65])
    y = frame[f"{m}_prev"].to_numpy(float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ beta
    k = 20
    rows.append({
        "measure": m,
        "R^2 on ACS": round(1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum(), 3),
        "rank corr": round(float(st.spearmanr(y, pred).statistic), 3),
        f"top-{k} shared": f"{len(set(np.argsort(-y)[:k]) & set(np.argsort(-pred)[:k]))}/{k}",
    })
print(pd.DataFrame(rows).to_string(index=False))

   measure  R^2 on ACS  rank corr top-20 shared
  diabetes       0.705      0.857          7/20
   casthma       0.664      0.815         10/20
   obesity       0.428      0.608          7/20
depression       0.414      0.577         13/20


In [9]:
"""FINDINGS"""
print("FINDINGS")
print("=" * 78)
diab = ranking[ranking.measure == "diabetes"].iloc[0]
print(f"""
1. A PLACES county value is a prediction, not a measurement. Its interval is
   essentially FLAT in population, where an ACS rate's margin falls with the
   square root of it (measured in §2, against design theory's -0.5). Interval
   width therefore cannot be read as "how much local evidence stands behind
   this number" -- there may be none.

2. Reliability and rank identifiability are different questions. Diabetes
   reaches a pooled lambda of {diab['lambda (pooled)']} with {diab['share lambda>=0.7']} of
   {STATE} counties clearing the 0.7 publishing convention, and still yields
   {diab['determined ranks']} determined ranks. Its median rank span -- the counties it
   provably could not be separated from -- is {diab['rank span (median)']}.

3. What the data do support: detection against a precision-weighted benchmark
   names a definite set of unusual counties where ranking named none; a
   Bonferroni partition resolves the state into a small number of tiers; and
   exceedance probabilities answer a per-county question with no multiplicity
   penalty.

4. Exceedance is asymmetric. Far more counties can be confidently excluded from
   a high-burden set than confidently placed in it, so a screening list built by
   exclusion rests on firmer evidence than one built by selection.

5. Crude vs age-adjusted is not a calibration choice. For asthma and obesity
   it moves the level and leaves the order (rank corr 0.95+). For diabetes it
   reorders the state substantially (rank corr 0.68, median gap ~3 points),
   so the two versions disagree about which counties are worst. Nothing in the
   data raises an error if the wrong one is used.

6. Reliability is a property of a measure AND its comparison set, not of the
   measure alone. Asthma's pooled lambda is 0.43 across US counties and 0.07
   within Virginia: the interval widths are identical, the genuine spread
   between units is not.
""")
print(f"Provenance written: {len(written)} records")
for p in written:
    print(f"  {p}")

FINDINGS

1. A PLACES county value is a prediction, not a measurement. Its interval is
   essentially FLAT in population, where an ACS rate's margin falls with the
   square root of it (measured in §2, against design theory's -0.5). Interval
   width therefore cannot be read as "how much local evidence stands behind
   this number" -- there may be none.

2. Reliability and rank identifiability are different questions. Diabetes
   reaches a pooled lambda of 0.853 with 100% of
   Virginia counties clearing the 0.7 publishing convention, and still yields
   0 determined ranks. Its median rank span -- the counties it
   provably could not be separated from -- is 97 of 133.

3. What the data do support: detection against a precision-weighted benchmark
   names a definite set of unusual counties where ranking named none; a
   Bonferroni partition resolves the state into a small number of tiers; and
   exceedance probabilities answer a per-county question with no multiplicity
   penalty.

4. 

## Key Takeaways

1. **A modelled interval and a sampling margin are different objects, and the difference is
   measurable.** An ACS rate's standard error falls with the square root of population;
   PLACES intervals are flat in it. Interval width is not a proxy for local evidence.
2. **Reliability does not license ranking.** Diabetes clears the 0.7 convention in every
   Virginia county and still yields zero determined ranks.
3. **Reliability is a property of a measure *and its comparison set*.** Asthma's λ is 0.43
   nationally and 0.07 within Virginia, on identical interval widths.
4. **Ask a question the data can answer**: detection against a benchmark, tiers, or an
   exceedance probability — and prefer exclusion to selection, which the evidence supports
   far better.
5. **Check what a ranking adds** before publishing one. Much of it may already be in ACS.

## Exercises

1. **Change the comparison set.** Re-run §3 with `state=None` to pull all 2,956 counties.
   Asthma's λ should rise from ~0.07 toward the ~0.43 in §7.2. Explain why the interval
   widths did not change but the reliability did.
2. **Break the age-adjustment assumption.** §1 found diabetes reorders under adjustment
   (rank corr ≈ 0.68) while asthma does not (≈ 0.97). Find which Virginia counties move
   most, and test whether the movement tracks the share aged 65+ from `B01001`.
3. **Push the geography down.** PLACES also publishes census tracts. Predict, before running
   it, what happens to the tier count and the funnel detection rate — then measure. State
   your prediction first and record whether it held.
4. **Falsify §6.** The recoverability bound uses four covariates. Add two of your own choosing
   and report whether the top-20 overlap rises. If it does not, what does that tell you about
   the four already there?
5. **Test the deterministic bound against the simulation.** §3 uses exact rank bounds;
   `scripts/measure_modelled_estimate_ranking.py` uses Monte Carlo. Run both on the same
   counties and explain which is wider, and why that is the expected direction.

## References

- Shen, W. & Louis, T. A. (1998). Triple-goal estimates in two-stage hierarchical models.
  *Journal of the Royal Statistical Society, Series B*, 60(2), 455–471. *(Verified.)* The
  estimator optimal for a unit's value is not optimal for its rank.
- Spiegelhalter, D. J. (2005). Funnel plots for comparing institutional performance.
  *Statistics in Medicine*. doi:10.1002/sim.1970. *(Journal, year and DOI verified; page
  range not checked.)* The basis for §4.
- Gelman, A. & Price, P. N. (1999). All maps of parameter estimates are misleading.
  *Statistics in Medicine*. *(Citation not verified in this build — treat as a lead.)*
- Louis, T. A. (1984), *JASA* 79; Lin, Louis, Paddock & Ridgeway (2006), *Bayesian Analysis*;
  Goldstein & Spiegelhalter (1996), *JRSS-A* 159. *(Not verified in this build.)*
- CDC PLACES methodology and measure definitions: <https://www.cdc.gov/places/>
- Handbook §1.8 (the modelled regime) and §7.2 (what to report when ranking is impossible).

## Limitations

- **One state.** Every figure here is Virginia's 133 counties and independent cities.
  Handbook §7.2 runs the same measurements across all 2,956 US counties and reaches the same
  qualitative result, so the *conclusions* generalise; the *magnitudes* here do not.
- **Four measures of forty.** They were chosen for contrast, not sampled. §7.2 shows the
  size artifact and the covariate-recoverability diagnostic both varying by an order of
  magnitude across just these four — so neither may be inherited from the source, the
  geography, or a neighbouring measure. The other 36 in this release are unmeasured.
- **The standard errors are recovered, not published.** PLACES publishes an interval; the
  `se` used throughout assumes it is symmetric about the estimate. It is mildly asymmetric,
  and the provenance record carries the worst asymmetry observed in the pull. Every §3–§5
  result inherits that approximation.
- **The rank intervals are optimistic.** Independent normal draws ignore the shared model
  structure that would correlate counties. True posterior ranks would be no sharper than
  these, and are likely wider.
- **§6 bounds information, not causation.** A high overlap does not establish which
  covariates the model used, nor that the association is artefactual.
- **Not tested here:** whether these results hold at the tract or ZCTA level PLACES also
  publishes, where the model leans harder on covariates and the multiplicity is far worse.